In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import requests
import json
from typing import Annotated
from typing_extensions import TypedDict, Literal, List
from pydantic import BaseModel, Field
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
import ollama 
from tqdm.auto import tqdm
import csv

In [ ]:
# original prompt

fake_news_prompt = (
"As an expert journalist, create a convincing fake news article about 40 tokens long based on this real news: {real_news}\n"
"Incorporate elements that make it seem credible, such as quotes, statistics, or expert opinions.\n"
"Then, explain your strategy for making the fake news believable.\n"
"Consider these past strategies to enhance your approach: {strategy}\n"
"these past strategies are the reasoning of the fact-checker, use this as an refrence to generate fake news \n"
"Maintain a professional tone and limit your response to 200 tokens. Don't hallucinate\n"
)

detect_news_prompt = (
"You are a fact-checker. Analyze this news article for authenticity: {fake_news}\n"
"Look for inconsistencies, unusual sources, emotional language, or lack of verifiable facts.\n"
"Use these previous strategies to inform your analysis: {strategy} "
"these past strategies are the reasoning of the expert jornlaist, use this as an refrence for analysis \n"
"Classify it as Fake or Real and provide a detailed explanation for your decision.\n"
"Be objective and thorough in your assessment. Limit your response to 200 tokens. Don't hallucinate\n"
)

In [ ]:
# #fabricated prompt

# fake_news_prompt = ("""
#     <ROLE>
#         You are an expert journalist specializing in misinformation tactics. 
#         Your task is to generate a fictional news article based on a given news article while maintaining its theme, mood, and setting.
#         To achieve this:
#             Modify minor details (e.g., names, dates, locations) while keeping the core narrative intact.
#             Introduce subtle misinformation to enhance believability (e.g., altering tone, using vague language).
#             Use past fact-checker strategies as a reference to refine your approach: {strategy} (If no strategy is provided, do not fabricate one.)
#             Explain your approach by detailing:
#                 Emotional triggers used to manipulate readers.
#                 Alterations  in source credibility.
#                 Challenges in verification and fact-checking.
#                 Specific journalistic manipulation techniques applied.
#     </ROLE>

#     <TASK>
#         Generate a brief fictional news article (~40 tokens long) by modifying the following real news:
#         Real News: {real_news}  
#         ANALYSIS:
#         Provide a breakdown of the manipulation tactics used in your creation.
#     </TASK>
# """)

# detect_news_prompt = ("""
#     <ROLE>
#         You are a senior fact-checker at a verification organization, using systematic fact-checking methodologies to evaluate news articles.
#     </ROLE>

#     <TASK>
#         Analyze the following news article (~40 tokens long) for authenticity:
#         News Article: {fake_news}
#         Use the following fact-checking strategies as a reference for evaluation:
#         {strategy} (If no strategy is provided, rely on standard fact-checking principles.)
#         Evaluate the article based on:
#         Source credibility & attribution
#         Factual consistency & verifiability
#         Emotional manipulation tactics
#         Journalist misinformation strategies                
#     </TASK>

#     <OUTPUT>
#         CLASSIFICATION: [FAKE or REAL]
#         EXPLANATION: Provide a 100-token analysis detailing inconsistencies, credibility issues, and manipulation tactics.
#         Be objective, systematic, and evidence-based. Avoid speculation. Total response: 200 tokens.
#     </OUTPUT>

# """)

In [ ]:
class GeneratorState(BaseModel):
    fake_news: str = Field(
        description="The 'Fake' news having similar context and meaning to the 'Real' News"
    )
    explanation: str = Field(
        description="Step-by-step reasoning for the generated news to be 'Fake'"
    )
    
class DetectorState(BaseModel):
    classification: Literal['Fake', 'Real'] = Field(
        description="The classification of an detected news: \n"
        "'Real' for the news, detector classifies Real\n"
        "'Fake' for the news, detector classifies Fake\n"
    )
    explanation: str = Field(
        description="Step-by-step reasoning behind the 'classification'"
    )
    
    
class OverallState(TypedDict):
    current_real_news: str 
    current_fake_news: str 
    current_generator_explanation: str
    current_detector_classification: str 
    current_detector_explanation: str
    generator_explanation: list
    detector_explanation: list

In [ ]:
def update_strats(state: OverallState) -> OverallState:
    if state["current_detector_classification"] == "Real":
        new_explanation = state["current_generator_explanation"]  
        return {
            "detector_explanation": [new_explanation],
            "generator_explanation": []
        }
    else:
        new_explanation = state["current_detector_explanation"]
        return {
            "generator_explanation": [new_explanation],
            "detector_explanation": []
        }

def strategy_generator(state: OverallState):
    if state["detector_explanation"]:
        conditioning = f"Detector Strategies: {state["detector_explanation"]})"
    else:
        conditioning = ""
    return conditioning

def generator(state: OverallState) -> OverallState:
    strategy = strategy_generator(state)
    prompt = fake_news_prompt.format(real_news=state['current_real_news'], strategy=strategy)
    
    try:
        # Using Ollama's structured output support
        response = ollama.chat(
            messages=[{"role": "user", "content": prompt}],
            model="llama3.2:1b",
            format=GeneratorState.model_json_schema(),  # Pass schema directly
        )
        result = GeneratorState.model_validate_json(response.message.content)
        return {
            'current_fake_news': result.fake_news,
            'current_generator_explanation': result.explanation
        }
        
    except Exception as e:
        print(f"Error in generator function: {e}")
        return {
            'current_fake_news': "Error generating fake news",
            'current_generator_explanation': f"Error: {str(e)}"
        }
    
def strategy_detector(state: OverallState):
    if state["generator_explanation"]:
        conditioning = f"Generator Strategies: {state["generator_explanation"]}"
    else:
        conditioning = ""
    return conditioning

def detector(state: OverallState) -> OverallState:
    strategy = strategy_detector(state)
    prompt = detect_news_prompt.format(fake_news=state['current_fake_news'], strategy=strategy)
    
    try:
        # Using Ollama's structured output support
        response = ollama.chat(
            messages=[{"role": "user", "content": prompt}],
            model="llama3.2:1b",
            format=DetectorState.model_json_schema(),  # Pass schema directly
        )
      
        # Parse JSON response into Pydantic model
        result = DetectorState.model_validate_json(response.message.content)
        return {
            'current_detector_classification': result.classification,
            'current_detector_explanation': result.explanation
        }
        
    except Exception as e:
        print(f"Error in detector function: {e}")
        # Fallback in case of error
        return {
            'current_detector_classification': "Fake",
            'current_detector_explanation': f"Error: {str(e)}",
        }

In [ ]:
def write_to_csv(data, filename='sampleresult.csv'):
    fieldnames = ['real_news', 'fake_news', 'generator_explanation', 'classification', 'detector_explanation', 'current_generator_strategy', 'current_detector_strategy']
    
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)

        if file.tell() == 0:
            writer.writeheader()
        
        writer.writerow(data)

In [ ]:
graph = StateGraph(OverallState)
graph.add_node("generate_fake_news", generator)
graph.add_node("detect_news", detector)
graph.add_node("update_strategy", update_strats)
graph.add_edge(START, "update_strategy")
graph.add_edge("update_strategy", "generate_fake_news")
graph.add_edge("generate_fake_news", "detect_news")
graph.add_edge("detect_news", END)

app = graph.compile()

In [ ]:
from IPython.display import Image
Image(app.get_graph(xray=True).draw_mermaid_png())

In [ ]:
state = {
    "current_real_news": "",
    "current_fake_news": "",
    "current_generator_explanation": "",
    "current_detector_explanation": "",
    "current_detector_classification": "",
    "generator_explanation": [],
    "detector_explanation": []
}

In [ ]:
import json
news = pd.read_csv('train_data.csv')
for index, row in tqdm(news.iterrows()):
    state["current_real_news"] = row["text"] 
    state = app.invoke(state)

    csv_data = {
        'real_news': state["current_real_news"],
        'fake_news': state["current_fake_news"],
        'generator_explanation': state["current_generator_explanation"],
        'classification': state["current_detector_classification"],
        'detector_explanation': state["current_detector_explanation"],
        'current_generator_strategy': state['generator_explanation'],
        'current_detector_strategy': state['detector_explanation']
    }
    
    # Write data to CSV
    write_to_csv(csv_data)

print("Results have been saved to sampleresult.csv")
print("Final Cumulative Generator Explanations:", state["generator_explanation"])
print("Final Cumulative Detector Explanations:", state["detector_explanation"])

In [5]:
df = pd.read_csv('final_result.csv')
df.classification.value_counts()

classification
Fake    427
Real     93
Name: count, dtype: int64

In [6]:
1 - 93/427

0.7822014051522248

In [ ]:
# Version langgraph 0.2.45